In [ ]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# load data
DATA_ROOT = Path(os.getenv("SCADA_ROOT"))

FARM = os.getenv("WIND_FARM_A")

ROOT = DATA_ROOT / FARM
EVENT_INFO_PATH = ROOT / "event_info.csv"
DATASETS_DIR = ROOT / "datasets"

events = pd.read_csv(EVENT_INFO_PATH, sep=";")

# Basic cleaning
events["event_start_id"] = pd.to_numeric(events["event_start_id"], errors="coerce")
events["event_end_id"]   = pd.to_numeric(events["event_end_id"], errors="coerce")
events = events.dropna(subset=["event_id", "event_start_id", "event_end_id"]).copy()

events["event_id"] = pd.to_numeric(events["event_id"], errors="coerce").astype("Int64")
events["start_id"] = events[["event_start_id", "event_end_id"]].min(axis=1).astype("Int64")
events["end_id"]   = events[["event_start_id", "event_end_id"]].max(axis=1).astype("Int64")

print("event_info rows:", len(events))
print("event_label counts:\n", events["event_label"].value_counts(dropna=False))
print("\nevent_info columns:", events.columns.tolist())

# Pick a sample event_id
sample_event_id = int(events["event_id"].dropna().iloc[0])
sample_path = DATASETS_DIR / f"{sample_event_id}.csv"
df = pd.read_csv(sample_path, sep=";")

print("\nSample event_id:", sample_event_id)
print("Sample file shape:", df.shape)
print("Sample columns match expected:", set(["time_stamp","asset_id","id","train_test"]).issubset(df.columns))

# Check id range vs event window
df["id"] = pd.to_numeric(df["id"], errors="coerce")
df = df.dropna(subset=["id"]).copy()
df["id"] = df["id"].astype(np.int64)

row = events.loc[events["event_id"] == sample_event_id].iloc[0]
start_id = int(row["start_id"])
end_id = int(row["end_id"])

print("\nSCADA id min/max:", df["id"].min(), df["id"].max())
print("Event window start/end:", start_id, end_id)
print("Window within file id range?:", df["id"].min() <= start_id <= df["id"].max() and df["id"].min() <= end_id <= df["id"].max())

print("\ntrain_test unique values:", df["train_test"].astype(str).str.lower().value_counts())

event_info rows: 22
event_label counts:
 event_label
anomaly    12
normal     10
Name: count, dtype: int64

event_info columns: ['asset', 'event_id', 'event_label', 'event_start', 'event_start_id', 'event_end', 'event_end_id', 'event_description', 'start_id', 'end_id']

Sample event_id: 68
Sample file shape: (54358, 86)
Sample columns match expected: True

SCADA id min/max: 0 54357
Event window start/end: 52063 54076
Window within file id range?: True

train_test unique values: train_test
train         52063
prediction     2295
Name: count, dtype: int64


In [4]:
# Label rows
df["in_event_window"] = (
    (df["id"] >= start_id) &
    (df["id"] <= end_id)
).astype(int)

# Count rows
rows_in_window = df["in_event_window"].sum()
expected_rows = end_id - start_id + 1

print("Rows inside window:", rows_in_window)
print("Expected window size:", expected_rows)
print("Match?:", rows_in_window == expected_rows)

# Sanity check overlap with prediction set
print("\nRows inside window that are labeled prediction:")
print(df.loc[df["in_event_window"] == 1, "train_test"].value_counts())

Rows inside window: 2014
Expected window size: 2014
Match?: True

Rows inside window that are labeled prediction:
train_test
prediction    2014
Name: count, dtype: int64


In [5]:
results = []

for event_id in events["event_id"]:
    try:
        df = pd.read_csv(DATASETS_DIR / f"{int(event_id)}.csv", sep=";")
        df["id"] = pd.to_numeric(df["id"], errors="coerce")
        df = df.dropna(subset=["id"])
        df["id"] = df["id"].astype(int)

        row = events.loc[events["event_id"] == event_id].iloc[0]
        start_id = int(row["start_id"])
        end_id = int(row["end_id"])

        id_min = df["id"].min()
        id_max = df["id"].max()

        # Check window inside ID range
        window_inside = (id_min <= start_id <= id_max) and (id_min <= end_id <= id_max)

        # Label window rows
        df["in_event_window"] = ((df["id"] >= start_id) & (df["id"] <= end_id)).astype(int)

        # Check if window only in prediction rows
        window_train_test = df.loc[df["in_event_window"] == 1, "train_test"].unique()
        window_only_prediction = set(window_train_test) == {"prediction"}

        results.append({
            "event_id": event_id,
            "window_inside_file": window_inside,
            "window_only_prediction": window_only_prediction,
            "window_size": df["in_event_window"].sum()
        })

    except Exception as ex:
        results.append({
            "event_id": event_id,
            "error": str(ex)
        })

validation_df = pd.DataFrame(results)
validation_df

,event_id,window_inside_file,window_only_prediction,window_size
0,68,True,True,2014
1,22,True,True,1005
2,72,True,True,1009
3,73,True,True,1009
4,0,True,True,2012
5,26,True,True,1009
6,40,True,True,4508
7,42,True,True,1007
8,10,True,True,981
9,45,True,True,1008


In [6]:
feature_counts = []

for event_id in events["event_id"]:
    df = pd.read_csv(DATASETS_DIR / f"{int(event_id)}.csv", sep=";")
    feature_counts.append({
        "event_id": event_id,
        "num_columns": df.shape[1]
    })

pd.DataFrame(feature_counts)

,event_id,num_columns
0,68,86
1,22,86
2,72,86
3,73,86
4,0,86
5,26,86
6,40,86
7,42,86
8,10,86
9,45,86


In [8]:
asset_counts = []

for event_id in events["event_id"]:
    df = pd.read_csv(DATASETS_DIR / f"{int(event_id)}.csv", sep=";")
    asset_counts.append(df["asset_id"].unique()[0])

print("Unique asset_ids:", set(asset_counts))
print("Number of unique turbines:", len(set(asset_counts)))

Unique asset_ids: {np.int64(0), np.int64(10), np.int64(11), np.int64(13), np.int64(21)}
Number of unique turbines: 5


In [7]:
df = pd.read_csv(DATASETS_DIR / f"{int(events['event_id'].iloc[0])}.csv", sep=";")

missing_pct = df.isna().mean().mean() * 100
print("Overall missing percentage in sample file:", missing_pct)

Overall missing percentage in sample file: 0.0


In [ ]:
master_parts = []
FARM = "Wind Farm A"

for event_id in events["event_id"]:
    df = pd.read_csv(DATASETS_DIR / f"{int(event_id)}.csv", sep=";")

    df["id"] = pd.to_numeric(df["id"], errors="coerce")
    df = df.dropna(subset=["id"])
    df["id"] = df["id"].astype(int)

    row = events.loc[events["event_id"] == event_id].iloc[0]
    start_id = int(row["start_id"])
    end_id = int(row["end_id"])

    df["in_event_window"] = ((df["id"] >= start_id) & (df["id"] <= end_id)).astype(int)

    df["event_id"] = event_id
    df["event_label"] = row["event_label"]
    df["farm"] = FARM

    master_parts.append(df)

master = pd.concat(master_parts, ignore_index=True)

master.shape

np.int64(38257)

In [14]:
summary_rows = []

for event_id, g in master.groupby("event_id"):
    
    summary_rows.append({
        "event_id": event_id,
        "asset_id": g["asset_id"].iloc[0],
        "event_label": g["event_label"].iloc[0],
        "total_rows": len(g),
        "train_rows": (g["train_test"] == "train").sum(),
        "prediction_rows": (g["train_test"] == "prediction").sum(),
        "window_rows": g["in_event_window"].sum()
    })

event_summary = pd.DataFrame(summary_rows)

event_summary.sort_values("window_rows", ascending=False)

,event_id,asset_id,event_label,total_rows,train_rows,prediction_rows,window_rows
11,40,10,anomaly,56158,51219,4939,4508
1,3,10,normal,55487,52185,3302,3014
3,13,21,normal,54010,50427,3583,3007
17,71,0,normal,54744,52295,2449,2017
16,69,11,normal,54813,52220,2593,2017
15,68,11,anomaly,54358,52063,2295,2014
0,0,0,anomaly,54986,52148,2838,2012
7,24,0,normal,55003,52289,2714,1995
5,17,10,normal,55090,52309,2781,1917
8,25,11,normal,54712,52289,2423,1847


In [15]:
meta_cols = {
    "time_stamp",
    "asset_id",
    "id",
    "train_test",
    "status_type_id",
    "event_id",
    "event_label",
    "farm",
    "in_event_window"
}

sensor_cols = [c for c in master.columns if c not in meta_cols]

len(sensor_cols)

81

In [16]:
mean_shift_results = []

for event_id, g in master.groupby("event_id"):
    
    train = g[g["train_test"] == "train"]
    window = g[g["in_event_window"] == 1]
    
    if len(train) == 0 or len(window) == 0:
        continue
    
    train_mean = train[sensor_cols].mean()
    window_mean = window[sensor_cols].mean()
    
    mean_shift = (window_mean - train_mean).abs()
    
    max_shift = mean_shift.max()
    
    mean_shift_results.append({
        "event_id": event_id,
        "event_label": g["event_label"].iloc[0],
        "asset_id": g["asset_id"].iloc[0],
        "max_mean_shift": max_shift
    })

mean_shift_df = pd.DataFrame(mean_shift_results)

mean_shift_df.sort_values("max_mean_shift", ascending=False)

,event_id,event_label,asset_id,max_mean_shift
21,92,normal,11,110943.918139
13,45,anomaly,13,93371.550884
15,68,anomaly,11,57778.889532
6,22,anomaly,21,56566.640168
5,17,normal,10,55878.429523
20,84,anomaly,13,44187.442288
18,72,anomaly,21,43042.528888
12,42,anomaly,10,42766.768430
9,26,anomaly,0,42580.946854
3,13,normal,21,39381.296174


In [17]:
z_shift_results = []

for event_id, g in master.groupby("event_id"):
    
    train = g[g["train_test"] == "train"]
    window = g[g["in_event_window"] == 1]
    
    if len(train) == 0 or len(window) == 0:
        continue
    
    train_mean = train[sensor_cols].mean()
    train_std = train[sensor_cols].std().replace(0, np.nan)
    
    window_mean = window[sensor_cols].mean()
    
    z_scores = ((window_mean - train_mean) / train_std).abs()
    
    max_z = z_scores.max()
    
    z_shift_results.append({
        "event_id": event_id,
        "event_label": g["event_label"].iloc[0],
        "asset_id": g["asset_id"].iloc[0],
        "max_z_shift": max_z
    })

z_shift_df = pd.DataFrame(z_shift_results)

z_shift_df.sort_values("max_z_shift", ascending=False)

,event_id,event_label,asset_id,max_z_shift
11,40,anomaly,10,2.394312
13,45,anomaly,13,1.392508
15,68,anomaly,11,1.349623
0,0,anomaly,0,1.342799
6,22,anomaly,21,1.336982
19,73,anomaly,0,1.271577
21,92,normal,11,1.234154
10,38,normal,13,1.079746
17,71,normal,0,0.984254
20,84,anomaly,13,0.968640


In [18]:
volatility_results = []

for event_id, g in master.groupby("event_id"):
    
    train = g[g["train_test"] == "train"]
    window = g[g["in_event_window"] == 1]
    
    if len(train) == 0 or len(window) == 0:
        continue
    
    train_std = train[sensor_cols].std().replace(0, np.nan)
    window_std = window[sensor_cols].std()
    
    vol_ratio = (window_std / train_std)
    
    max_vol_ratio = vol_ratio.max()
    
    volatility_results.append({
        "event_id": event_id,
        "event_label": g["event_label"].iloc[0],
        "asset_id": g["asset_id"].iloc[0],
        "max_volatility_ratio": max_vol_ratio
    })

volatility_df = pd.DataFrame(volatility_results)

volatility_df.sort_values("max_volatility_ratio", ascending=False)

,event_id,event_label,asset_id,max_volatility_ratio
11,40,anomaly,10,2.885627
20,84,anomaly,13,1.839875
13,45,anomaly,13,1.726383
10,38,normal,13,1.600949
19,73,anomaly,0,1.582459
21,92,normal,11,1.577797
17,71,normal,0,1.448161
15,68,anomaly,11,1.447011
5,17,normal,10,1.443137
2,10,anomaly,10,1.408867


In [19]:
# Merge the two metrics
combined = z_shift_df.merge(volatility_df, on=["event_id", "event_label", "asset_id"])

# Compute composite score
combined["severity_score"] = combined["max_z_shift"] + np.log(combined["max_volatility_ratio"])

combined.sort_values("severity_score", ascending=False)

,event_id,event_label,asset_id,max_z_shift,max_volatility_ratio,severity_score
11,40,anomaly,10,2.394312,2.885627,3.454054
13,45,anomaly,13,1.392508,1.726383,1.938537
19,73,anomaly,0,1.271577,1.582459,1.730558
15,68,anomaly,11,1.349623,1.447011,1.719123
21,92,normal,11,1.234154,1.577797,1.690184
0,0,anomaly,0,1.342799,1.373419,1.660102
20,84,anomaly,13,0.968640,1.839875,1.578338
10,38,normal,13,1.079746,1.600949,1.550343
6,22,anomaly,21,1.336982,1.211882,1.529156
17,71,normal,0,0.984254,1.448161,1.354549


In [20]:
combined.groupby("event_label")["severity_score"].describe()

,count,mean,std,min,25%,50%,75%,max
event_label,,,,,,,,
anomaly,12.0,1.615917,0.647771,0.983510,1.206463,1.553747,1.721982,3.454054
normal,10.0,0.933387,0.488439,0.428622,0.492453,0.787489,1.323524,1.690184


In [21]:
top_event_id = 40

g = master[master["event_id"] == top_event_id]

train = g[g["train_test"] == "train"]
window = g[g["in_event_window"] == 1]

train_mean = train[sensor_cols].mean()
train_std = train[sensor_cols].std().replace(0, np.nan)
window_mean = window[sensor_cols].mean()

z_scores = ((window_mean - train_mean) / train_std).abs()

top_sensors = z_scores.sort_values(ascending=False).head(10)

top_sensors

sensor_41_avg    2.394312
sensor_6_avg     1.301025
sensor_5_min     1.173371
sensor_53_avg    1.171659
sensor_0_avg     1.148920
sensor_5_avg     1.144331
sensor_43_avg    1.037487
sensor_10_avg    0.982875
sensor_12_avg    0.961659
sensor_5_max     0.919795
dtype: float64

In [22]:
normal_event_id = 92

g = master[master["event_id"] == normal_event_id]

train = g[g["train_test"] == "train"]
window = g[g["in_event_window"] == 1]

train_mean = train[sensor_cols].mean()
train_std = train[sensor_cols].std().replace(0, np.nan)
window_mean = window[sensor_cols].mean()

z_scores_normal = ((window_mean - train_mean) / train_std).abs()

z_scores_normal.sort_values(ascending=False).head(10)

power_30_min     1.234154
power_29_min     1.115931
sensor_40_avg    1.090433
sensor_15_avg    1.025308
sensor_23_avg    1.023627
sensor_45        1.016207
power_30_avg     1.015228
sensor_50        1.015224
sensor_24_avg    1.009962
sensor_25_avg    1.004176
dtype: float64

In [23]:
concentration_results = []

for event_id, g in master.groupby("event_id"):
    
    train = g[g["train_test"] == "train"]
    window = g[g["in_event_window"] == 1]
    
    train_mean = train[sensor_cols].mean()
    train_std = train[sensor_cols].std().replace(0, np.nan)
    window_mean = window[sensor_cols].mean()
    
    z_scores = ((window_mean - train_mean) / train_std).abs()
    
    concentration_results.append({
        "event_id": event_id,
        "event_label": g["event_label"].iloc[0],
        "max_z": z_scores.max(),
        "num_sensors_z_gt_1": (z_scores > 1).sum(),
        "num_sensors_z_gt_1_5": (z_scores > 1.5).sum()
    })

concentration_df = pd.DataFrame(concentration_results)

concentration_df.sort_values("max_z", ascending=False)

,event_id,event_label,max_z,num_sensors_z_gt_1,num_sensors_z_gt_1_5
11,40,anomaly,2.394312,7,1
13,45,anomaly,1.392508,6,0
15,68,anomaly,1.349623,3,0
0,0,anomaly,1.342799,8,0
6,22,anomaly,1.336982,8,0
19,73,anomaly,1.271577,4,0
21,92,normal,1.234154,11,0
10,38,normal,1.079746,2,0
17,71,normal,0.984254,0,0
20,84,anomaly,0.968640,0,0


In [24]:
energy_results = []

for event_id, g in master.groupby("event_id"):
    
    train = g[g["train_test"] == "train"]
    window = g[g["in_event_window"] == 1]
    
    train_mean = train[sensor_cols].mean()
    train_std = train[sensor_cols].std().replace(0, np.nan)
    window_mean = window[sensor_cols].mean()
    
    z_scores = ((window_mean - train_mean) / train_std)
    
    energy = np.sqrt(np.nansum(z_scores**2))
    
    energy_results.append({
        "event_id": event_id,
        "event_label": g["event_label"].iloc[0],
        "energy_score": energy
    })

energy_df = pd.DataFrame(energy_results)

energy_df.sort_values("energy_score", ascending=False)

,event_id,event_label,energy_score
21,92,normal,5.543058
11,40,anomaly,5.427590
13,45,anomaly,5.418149
6,22,anomaly,5.230898
18,72,anomaly,5.012433
15,68,anomaly,4.519396
0,0,anomaly,4.477815
2,10,anomaly,4.218361
9,26,anomaly,4.215925
5,17,normal,4.115571
